# Phase 8: Product-Level Trust Aggregation & Evaluation

Aggregate review-level trust scores to product level and evaluate ranking quality.

**Key Metrics:**
- Trust-weighted rating formula
- NDCG@K (Normalized Discounted Cumulative Gain)
- Precision@K
- Comparison vs baselines (raw average, count-weighted)

In [29]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## 1. Load Data

In [30]:
df = pd.read_csv("../data/processed/reviews_with_predicted_trust.csv")

print(f"Dataset shape: {df.shape}")
print(f"Predicted trust score stats:")
print(df['predicted_trust_score'].describe())

Dataset shape: (719967, 8)
Predicted trust score stats:
count    719967.000000
mean          0.571692
std           0.108702
min           0.146959
25%           0.513470
50%           0.567607
75%           0.612114
max           0.995795
Name: predicted_trust_score, dtype: float64


## 2. Product-Level Trust Aggregation

In [31]:
df['weighted_rating'] = df['predicted_trust_score'] * df['rating']

product_scores = df.groupby('product_id').agg({
    'weighted_rating': 'sum',
    'predicted_trust_score': 'sum',
    'rating': ['mean', 'count', 'std']
}).reset_index()

product_scores.columns = ['product_id', 'weighted_rating_sum', 'trust_sum', 
                           'avg_rating', 'review_count', 'rating_std']

product_scores['trust_weighted_rating'] = (
    product_scores['weighted_rating_sum'] / product_scores['trust_sum']
)

print(f"Product scores shape: {product_scores.shape}")
print(f"Trust-weighted rating stats:")
print(product_scores['trust_weighted_rating'].describe())

Product scores shape: (168281, 7)
Trust-weighted rating stats:
count    168281.000000
mean          3.784084
std           1.277954
min           1.000000
25%           3.000000
50%           4.000000
75%           5.000000
max           5.000000
Name: trust_weighted_rating, dtype: float64


## 3. Baseline Comparisons

In [32]:
product_scores['baseline_avg_rating'] = product_scores['avg_rating']

min_reviews = product_scores['review_count'].min()
max_reviews = product_scores['review_count'].max()
product_scores['review_weight'] = (
    (product_scores['review_count'] - min_reviews) / (max_reviews - min_reviews)
)
product_scores['baseline_count_weighted'] = (
    product_scores['avg_rating'] * (0.5 + 0.5 * product_scores['review_weight'])
)

print("Baseline scores computed")
print(f"Raw avg rating:        {product_scores['baseline_avg_rating'].mean():.3f} ± {product_scores['baseline_avg_rating'].std():.3f}")
print(f"Count-weighted:        {product_scores['baseline_count_weighted'].mean():.3f} ± {product_scores['baseline_count_weighted'].std():.3f}")
print(f"Trust-weighted:        {product_scores['trust_weighted_rating'].mean():.3f} ± {product_scores['trust_weighted_rating'].std():.3f}")

Baseline scores computed
Raw avg rating:        3.780 ± 1.274
Count-weighted:        1.891 ± 0.638
Trust-weighted:        3.784 ± 1.278


## 4. Ranking Metrics

In [33]:
def dcg_at_k(scores, k=10):
    scores = np.asarray(scores)[:k]
    if len(scores) == 0:
        return 0.0
    return np.sum(scores / np.log2(np.arange(2, len(scores) + 2)))

def ndcg_at_k(y_true, y_pred, k=10):
    sorted_indices = np.argsort(y_pred)[::-1]
    y_true_sorted = y_true[sorted_indices]
    dcg = dcg_at_k(y_true_sorted, k)
    y_true_sorted_ideal = np.sort(y_true)[::-1]
    idcg = dcg_at_k(y_true_sorted_ideal, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg

def precision_at_k(y_true, y_pred, k=10, threshold=4.0):
    sorted_indices = np.argsort(y_pred)[::-1][:k]
    y_true_at_k = y_true[sorted_indices]
    return np.mean(y_true_at_k >= threshold)

print("Ranking metrics defined")

Ranking metrics defined


In [34]:
y_true = product_scores['avg_rating'].values
results_ranking = []
k_values = [5, 10, 20]

for k in k_values:
    ndcg_trust = ndcg_at_k(y_true, product_scores['trust_weighted_rating'].values, k)
    prec_trust = precision_at_k(y_true, product_scores['trust_weighted_rating'].values, k)
    ndcg_avg = ndcg_at_k(y_true, product_scores['baseline_avg_rating'].values, k)
    prec_avg = precision_at_k(y_true, product_scores['baseline_avg_rating'].values, k)
    ndcg_count = ndcg_at_k(y_true, product_scores['baseline_count_weighted'].values, k)
    prec_count = precision_at_k(y_true, product_scores['baseline_count_weighted'].values, k)
    
    results_ranking.append({
        'K': k,
        'NDCG_Trust': ndcg_trust,
        'NDCG_Avg': ndcg_avg,
        'NDCG_Count': ndcg_count,
        'Prec_Trust': prec_trust,
        'Prec_Avg': prec_avg,
        'Prec_Count': prec_count
    })

ranking_results_df = pd.DataFrame(results_ranking)
print("\n" + "="*80)
print("RANKING METRICS COMPARISON")
print("="*80)
print(ranking_results_df.to_string(index=False))
print("="*80)
ranking_results_df.to_csv('../results/reports/ranking_metrics.csv', index=False)


RANKING METRICS COMPARISON
 K  NDCG_Trust  NDCG_Avg  NDCG_Count  Prec_Trust  Prec_Avg  Prec_Count
 5         1.0       1.0    0.917635         1.0       1.0         1.0
10         1.0       1.0    0.902837         1.0       1.0         1.0
20         1.0       1.0    0.897036         1.0       1.0         1.0


## 5. Save Final Product Scores

In [35]:
output_df = product_scores[[
    'product_id', 'review_count', 'avg_rating', 'rating_std',
    'baseline_avg_rating', 'baseline_count_weighted', 'trust_weighted_rating'
]].copy()

output_df.columns = [
    'product_id', 'review_count', 'avg_rating', 'rating_std',
    'score_raw_avg', 'score_count_weighted', 'score_trust_weighted'
]

output_df = output_df.sort_values('score_trust_weighted', ascending=False)
output_df.to_csv('../data/processed/product_trust_scores.csv', index=False)

print(f"Product scores saved: {output_df.shape[0]} products")
print(f"\nTop 20 products by trust-weighted score:")
print(output_df.head(20).to_string(index=False))

Product scores saved: 168281 products

Top 20 products by trust-weighted score:
product_id  review_count  avg_rating  rating_std  score_raw_avg  score_count_weighted  score_trust_weighted
B00842G61I             2         5.0         0.0            5.0              2.500657                   5.0
B01DX82JWC             2         5.0         0.0            5.0              2.500657                   5.0
B0012HPFXW             2         5.0         0.0            5.0              2.500657                   5.0
B00WP27XCO             1         5.0         NaN            5.0              2.500000                   5.0
B0012KYB0W             2         5.0         0.0            5.0              2.500657                   5.0
B01CQM3N58             1         5.0         NaN            5.0              2.500000                   5.0
B01FJSB1NC             3         5.0         0.0            5.0              2.501314                   5.0
B000GI49BM             5         5.0         0.0        

## 6. Summary

**Phase 8 Complete:**
- Aggregated review-level trust scores to product level
- Computed trust-weighted ratings
- Evaluated ranking quality with NDCG@K and Precision@K
- Compared against baselines
- Generated final product trust scores